# 2. Modeling DeepLearning LSTM and GRU

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
df = pd.read_csv("../dataset/cleaned_dataset_deeplearning_lstm_gru_model.csv")
df.head()

In [ ]:
# Tokenisasi
max_words = 5000
max_len = 50

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned_tweet'])
sequences = tokenizer.texts_to_sequences(df['cleaned_tweet'])
X = pad_sequences(sequences, maxlen=max_len)

In [ ]:
X

In [ ]:
# Label encoding
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['sentimen'])

In [ ]:
y

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

## 2.1 LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model_lstm = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model_lstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_lstm.summary()

# Train LSTM
history_lstm = model_lstm.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=32)

## 2.2 GRU

In [ ]:
from tensorflow.keras.layers import GRU

model_gru = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    GRU(64, return_sequences=False),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model_gru.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_gru.summary()

# Train GRU
history_gru = model_gru.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5, batch_size=32)

## 2.3 Evaluasi & Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Prediksi LSTM
y_pred_lstm = model_lstm.predict(X_test)
y_pred_lstm_labels = np.argmax(y_pred_lstm, axis=1)

In [ ]:
# Prediksi GRU
y_pred_gru = model_gru.predict(X_test)
y_pred_gru_labels = np.argmax(y_pred_gru, axis=1)


In [ ]:
# Report LSTM
print("LSTM Classification Report:")
print(classification_report(y_test, y_pred_lstm_labels, target_names=label_encoder.classes_))

In [ ]:
# Report GRU
print("GRU Classification Report:")
print(classification_report(y_test, y_pred_gru_labels, target_names=label_encoder.classes_))

In [ ]:
# Confusion Matrix GRU
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(y_test, y_pred_lstm_labels), annot=True, fmt='d', cmap='Oranges', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title("Confusion Matrix - LSTM")
plt.xlabel("Predicted")
plt.ylabel("Actual")

In [ ]:
plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(y_test, y_pred_gru_labels), annot=True, fmt='d', cmap='Purples', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title("Confusion Matrix - GRU")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
lr_conf_matrix = confusion_matrix(y_test, y_pred_lstm_labels)
gr_conf_matrix = confusion_matrix(y_test, y_pred_gru_labels)

In [ ]:
# Plot confusion matrix untuk Naive Bayes
plt.figure(figsize=(12, 5))

# Plot confusion matrix untuk Logistic Regression
plt.subplot(1, 2, 1)
sns.heatmap(lr_conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Negatif', 'Netral', 'Positif'], yticklabels=['Negatif', 'Netral', 'Positif'])
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.subplot(1, 2, 2)
sns.heatmap(gr_conf_matrix, annot=True, fmt='d', cmap='Reds', xticklabels=['Negatif', 'Netral', 'Positif'], yticklabels=['Negatif', 'Netral', 'Positif'])
plt.title("Confusion Matrix - Naive Bayes")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()